# Does distance grow as two images diverge higher up the tree?

**Question.** For each RDM, does mean distance rise as the two images of a pair share fewer ancestors in the a priori (Kiani-Mur) tree?

**Why it matters.** Rank agreement with the tree (`2_rank_agreement`) is one number per RDM and says nothing about *where* in the tree an RDM agrees. The depth profile is the shape behind that number: an RDM can order the coarse splits correctly yet be flat among deep relatives, or the reverse. Here the gradient is shown per LCA bin, condensed into one rank slope with uncertainty, and checked within branch-depth strata.

**RDM set.** `sem_km` (the tree itself, the reference), `sem_wn`, `clip_pre`, `clip_post`, `sens_pre`, `sens_post`, `behav_joint` (raw, ~13% of pairs unobserved and ignored) and, once notebook 1 has built them, its two SMACOF embeddings `behav_joint_mds_low` (stress-elbow dimensionality) and `behav_joint_mds_high` (95% of the held-out gain). LCA depth is pooled into the bins {0, 1, 2, 3, >=4} (0 = the pair diverges at the root), see `0_tree_shape`. Uncertainty is a 95% image bootstrap (1000 draws, stimuli resampled, cached).

**Run from this directory.**

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s2_semantic_hierarchy/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from analysis.s2_semantic_hierarchy import common as c
from analysis.s2_semantic_hierarchy import depth_gradient as dg

N_BOOT = 1000
CACHE = c.cache_path("depth_gradient")
try:
    RDMS = c.load_rdm_set(c.FULL_SET)
except FileNotFoundError:
    print(f"the behav_joint_mds embeddings are not built yet: falling back to the raw set {c.RAW_SET}")
    RDMS = c.load_rdm_set(c.RAW_SET)
PARTS = c.tree_parts()
RESULTS = dg.depth_gradient(RDMS, PARTS, n_boot=N_BOOT, cache_dir=CACHE)
print(f"{len(RDMS)} RDMs, {len(PARTS)} images; behav_joint has {np.isnan(RDMS[c.BEHAV]).mean():.1%} unobserved pairs")

9 RDMs, 725 images; behav_joint has 13.1% unobserved pairs


## 3.1 Normalised depth profile

**Question.** What is each RDM's mean distance per LCA bin, relative to its own mean in bin 0?

Dividing by bin 0 puts pixel Euclidean, cosine, WordNet and SpAM distances on one scale. A curve that falls from left to right means the RDM places hierarchically closer images closer together. `sem_km` is dashed and is the reference; the shaded bands are 95% image-bootstrap intervals (the LCA bin of each pair is resampled with the images). Tick labels give the number of pairs in each bin.

In [2]:
profile, lower, upper = dg.profile_tables(RESULTS, PARTS)
dg.plot_depth_profile(profile, lower, upper).show()
print(profile.round(3).to_string())

         n_pairs  sem_km  sem_wn  clip_pre  clip_post  sens_pre  sens_post  behav_joint  behav_joint_mds_low  behav_joint_mds_high
lca_bin                                                                                                                           
0         127246   1.000   1.000     1.000      1.000     1.000      1.000        1.000                1.000                 1.000
1          64696   0.698   1.012     0.912      0.928     0.979      1.005        0.955                0.953                 0.956
2          49272   0.370   0.810     0.853      0.869     0.984      1.001        0.720                0.633                 0.726
3          16130   0.227   0.568     0.628      0.665     0.905      0.911        0.441                0.369                 0.501
>=4         5106   0.160   0.283     0.471      0.500     0.839      0.856        0.248                0.254                 0.367


**Reading the curves.** `sem_km` falls to 0.16 of its bin-0 value in the deepest bin, a steep but not perfectly regular profile because the tree is ragged (see `0_tree_shape`). `behav_joint` is the steepest of the non-reference RDMs: 0.72 at LCA 2, 0.44 at LCA 3 and 0.25 at LCA >=4 (CI 0.23 to 0.27), against 0.28 for WordNet and 0.47 for CLIP pre in the same bin. `clip_pre` and `clip_post` fall gently and almost identically (0.47 and 0.50 in the deepest bin). The pixel RDMs are nearly flat (0.84 and 0.86 in the deepest bin) and stay near 1 up to LCA 2.

The two MDS embeddings of `behav_joint` are steeper in the middle bins (0.63 and 0.73 at LCA 2) and end at 0.25 (`mds_low`, d=5) and 0.37 (`mds_high`, d=20) in the deepest bin, so the embedding keeps the gradient and the high-d one flattens the deep end slightly.

Bin 0 holds 127,246 pairs (48% of the total) and the deepest bin only 5,106, so the right end of each curve rests on few pairs; the bands reflect image sampling only.

## 3.2 Monotonicity, root/deepest ratio and rank slope

**Question.** Is the gradient monotone, how steep is it, and how can it be summarised as one number with uncertainty?

* **Monotone falling**: the normalised profile never rises from one bin to the next.
* **Root / deepest ratio**: mean distance in bin 0 divided by that in the deepest bin (higher = steeper). It uses two bins only.
* **Rank slope**: minus the Spearman rho between a pair's distance and its LCA bin, over the pair's observed values. LCA bin grows as two images share deeper ancestors, so a distance that falls with depth has a *negative* rho. The slope is **sign-flipped**, so **positive = distance grows as pairs diverge higher up the tree**. It uses every pair, not only the bin means, and is the quantity `1_mds_dimensionality` reuses through `rank_slope_table()`.

In [3]:
table = dg.monotonicity_table(RESULTS)
print(table.round(3).to_string())
dg.plot_rank_slopes(table).show()

                      root_deepest_ratio  ratio_low  ratio_high  monotone_falling  rank_slope  slope_low  slope_high
rdm                                                                                                                 
sem_km                             6.247      5.908       6.626              True       0.907      0.898       0.917
sem_wn                             3.537      2.743       4.710             False       0.279      0.245       0.317
clip_pre                           2.125      2.010       2.251              True       0.434      0.399       0.469
clip_post                          2.002      1.903       2.118              True       0.398      0.366       0.432
sens_pre                           1.192      1.159       1.231             False       0.107      0.079       0.139
sens_post                          1.168      1.136       1.206             False       0.086      0.059       0.117
behav_joint                        4.030      3.759       4.323 

**Reading the table.** `sem_km` has a rank slope of 0.907 (CI 0.898 to 0.917); it is below 1 only because the tree is ragged and ties are many. Among the other RDMs `behav_joint` (0.393, CI 0.376 to 0.410), `clip_pre` (0.434, 0.399 to 0.469) and `clip_post` (0.398, 0.366 to 0.432) sit together with overlapping intervals: the rank slope does not separate behavior from CLIP. `sem_wn` is lower (0.279, 0.245 to 0.317) and the pixel RDMs are near zero (`sens_pre` 0.107, `sens_post` 0.086).

The root/deepest ratio orders the RDMs differently: `behav_joint` (4.03, CI 3.76 to 4.32) is far steeper than `clip_pre` (2.13), because the ratio looks at the two end bins only, where behavior falls much further, while the rank slope weighs all pairs. The embeddings are the steepest non-reference RDMs on rank slope: `behav_joint_mds_low` 0.625 (CI 0.602 to 0.648) and `behav_joint_mds_high` 0.686 (0.662 to 0.709), against 0.393 for the raw RDM; their intervals do not overlap the raw one, and both clear CLIP. This agrees with notebook 1, which reports 0.63 at d=5 and 0.69 at d=20. In the ratio, `mds_low` (3.94, CI 3.72 to 4.18) matches the raw RDM and `mds_high` is lower (2.72, 2.58 to 2.90). Both embeddings are monotone falling. WordNet's ratio (3.54) is the least certain (CI 2.74 to 4.71). Monotone falling holds for `sem_km`, both CLIP RDMs and `behav_joint`. It fails for `sem_wn` (bin 1 is 1.012, above bin 0), for `sens_post` (bin 1 is 1.005, above bin 0) and for `sens_pre` (bin 2 is above bin 1).

## 3.3 Depth profiles within branch-depth strata

**Question.** Does the gradient survive when pairs are compared only with pairs whose images sit at the same tree depths?

The tree is ragged, so the stratum (the pair of pooled leaf depths: 3x3, 3x>=4, >=4x>=4) fixes which LCA bins can occur. Some bins are therefore empty: 3x3 has no LCA 0 pairs and no LCA >=4 pairs; 3x>=4 has no LCA >=4 pairs. Each stratum's curve is **normalised to its own first non-empty bin** (listed in the table below), so curves are comparable in shape, not in level, across strata. Subplots share the y axis.

**Domain confound.** Every animate image is deep (>=4) and every depth-3 image is inanimate (`0_tree_shape`), so 3x3 is inanimate-only and its pairs never diverge at the root. A stratum difference partly reflects the animate/inanimate split and is not a clean depth effect.

In [4]:
print(dg.strata_table(RESULTS, PARTS).to_string())
dg.plot_strata_profiles(RESULTS, PARTS).show()

              0      1      2     3   >=4 normalised_to_bin
stratum                                                    
3x3           0  21983  27596  6701     0                 1
3x>=4    100128  19317  11059   200     0                 0
>=4x>=4   27118  23396  10617  9229  5106                 0


In [5]:
tb = dg.tree_bins(PARTS)
for k, stratum in enumerate(tb.strata):
    frame = pd.DataFrame({n: r.strata_profile[k] for n, r in RESULTS.items()}, index=tb.labels)
    print(f"\n{stratum}, normalised to LCA bin {tb.labels[tb.stratum_ref[k]]}")
    print(frame.dropna(how="all").round(3).to_string())


3x3, normalised to LCA bin 1
   sem_km  sem_wn  clip_pre  clip_post  sens_pre  sens_post  behav_joint  behav_joint_mds_low  behav_joint_mds_high
1    1.00   1.000     1.000      1.000     1.000      1.000        1.000                1.000                 1.000
2    0.50   0.841     0.972      0.967     0.998      0.981        0.827                0.725                 0.833
3    0.25   0.685     0.680      0.700     0.962      0.948        0.542                0.469                 0.633

3x>=4, normalised to LCA bin 0
   sem_km  sem_wn  clip_pre  clip_post  sens_pre  sens_post  behav_joint  behav_joint_mds_low  behav_joint_mds_high
0   1.000   1.000     1.000      1.000     1.000      1.000        1.000                1.000                 1.000
1   0.690   1.076     0.909      0.921     0.929      0.992        0.937                0.944                 0.937
2   0.414   0.832     0.817      0.839     0.925      0.972        0.733                0.670                 0.737
3   0.138 

**Reading the strata.** The gradient is present in each stratum with enough pairs.

* **3x3** (inanimate only, 56,280 pairs, bins 1 to 3): `behav_joint` falls 1.00, 0.83, 0.54, `clip_pre` 1.00, 0.97, 0.68 and `sem_wn` 1.00, 0.84, 0.69; the pixel RDMs stay near 1 until bin 3 (0.96 and 0.95). `sem_km` is 1.00, 0.50, 0.25.
* **3x>=4** (bins 0 to 3): `behav_joint` falls to 0.50 in bin 3 and `sem_wn` to 0.58, but both CLIP RDMs turn back up in bin 3 (0.90 and 0.91 after 0.82 and 0.84 in bin 2). Bin 3 holds only 200 pairs here, so this should not be over-read.
* **MDS embeddings**: in 3x3 `mds_low` falls 1.00, 0.73, 0.47 and `mds_high` 1.00, 0.83, 0.63; in 3x>=4 bin 3 they are 0.34 and 0.58 (200 pairs), with no CLIP-like rise. In >=4x>=4 they end at 0.26 and 0.37.
* **>=4x>=4** (all five bins, 75,466 pairs): `behav_joint` falls 1.00, 1.00, 0.59, 0.40, 0.25 and `clip_pre` 1.00, 0.94, 0.83, 0.65, 0.48. The pixel RDMs rise in bins 1 and 2 (1.05 and 1.09 for `sens_pre`) before falling.

Because the 3x3 stratum contains only inanimate pairs, the falls of `behav_joint` and CLIP there show that the gradient is not produced by the animate/inanimate split alone.

The primate drill-down of the old notebook is dropped: pooling depths 4 and 5 removed the bin it explained (see `0_tree_shape`).

## 3.4 Takeaways

1. **Behavior has a clear depth gradient.** `behav_joint` falls from 1.00 (LCA 0) to 0.25 (LCA >=4, CI 0.23 to 0.27) and is monotone; its rank slope is 0.393 (CI 0.376 to 0.410).
2. **Behavior and CLIP have the same overall rank slope but different shapes.** CLIP pre 0.434 (0.399 to 0.469) and CLIP post 0.398 (0.366 to 0.432) overlap with behavior; behavior is steeper in the root/deepest ratio (4.03 against 2.13 and 2.00), while CLIP is flatter at the deep end (0.47 and 0.50 in the deepest bin).
3. **The MDS embeddings sharpen the gradient.** Rank slope 0.625 (CI 0.602 to 0.648) at d=5 and 0.686 (0.662 to 0.709) at d=20, against 0.393 for the raw RDM; both are monotone, with deepest-bin values 0.25 and 0.37 and root/deepest ratios 3.94 and 2.72.
4. **WordNet is weaker and not monotone** (rank slope 0.279, CI 0.245 to 0.317; bin 1 is 1.012, above bin 0). Pixels are close to flat (rank slopes 0.107 and 0.086, deepest bin 0.84 and 0.86).
5. **The reference itself is not at 1** (rank slope 0.907) because the tree is ragged and tied.
6. **Within strata the gradient holds** for behavior (raw and embedded) and CLIP among inanimate-only 3x3 pairs (raw behavior 1.00, 0.83, 0.54; mds_low 1.00, 0.73, 0.47), so it is not only the animate/inanimate split. The pixel curves rise in bins 1 and 2 of the >=4x>=4 stratum.

**Limitations.** The deepest bins are small (5,106 pairs overall; 200 in 3x>=4 bin 3), the strata are confounded with domain, and normalising within each stratum fixes shape, not level, so strata cannot be compared on height. The bootstrap resamples images only, not participants. The embedding dimensionalities (5 and 20) were chosen in notebook 1 without the tree.